
# `halohalo` Preliminary Dataset Analysis

This notebook performs a first-pass exploratory analysis of `sapinsapin/halohalo` with one goal in mind: **decide how safe and useful this corpus is for adapting a Llama 3.1B model**.

Focus areas:
- corpus scale and split structure
- language and source balance
- sequence length pressure for training
- duplication, leakage, and scraping noise
- concrete implications for continued pretraining vs. instruction tuning

> Note: this notebook reads the locally cached Hugging Face Arrow files directly so it can run even when the `datasets` + `pandas` stack is unavailable in the environment.


In [6]:

from __future__ import annotations

from collections import Counter, defaultdict
from pathlib import Path
from urllib.parse import urlparse
import math
import re

import numpy as np
import pyarrow as pa
import pyarrow.ipc as ipc
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Markdown, display

DATASET_ROOT = Path.home() / ".cache" / "huggingface" / "datasets" / "sapinsapin___halohalo" / "default"


def find_dataset_dir(root: Path = DATASET_ROOT) -> Path:
    candidates = sorted(root.rglob("halohalo-train.arrow"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(
            "Could not find cached Arrow files for `sapinsapin/halohalo` under "
            f"{root}. Download the dataset once, then rerun this notebook."
        )
    return candidates[0].parent


def load_arrow_table(path: Path) -> pa.Table:
    with pa.memory_map(str(path), "r") as source:
        return ipc.open_stream(source).read_all()


def pct(part: float, whole: float) -> float:
    return 0.0 if whole == 0 else 100.0 * part / whole


def summarize_lengths(values: list[int]) -> dict[str, float]:
    arr = np.array(values, dtype=np.int64)
    return {
        "count": int(arr.size),
        "sum": int(arr.sum()),
        "mean": float(arr.mean()),
        "median": float(np.median(arr)),
        "p90": float(np.quantile(arr, 0.90)),
        "p95": float(np.quantile(arr, 0.95)),
        "max": int(arr.max()),
        "gt_1024": int((arr > 1024).sum()),
        "gt_2048": int((arr > 2048).sum()),
        "gt_4096": int((arr > 4096).sum()),
    }


def table_figure(title: str, headers: list[str], rows: list[list[object]], width: int = 950, height: int | None = None) -> go.Figure:
    height = height or max(320, 46 * (len(rows) + 1))
    fig = go.Figure(
        data=[
            go.Table(
                header=dict(values=headers, fill_color="#16324f", font=dict(color="white", size=12), align="left"),
                cells=dict(values=list(map(list, zip(*rows))) if rows else [[] for _ in headers], align="left", height=28),
            )
        ]
    )
    fig.update_layout(title=title, width=width, height=height, margin=dict(l=10, r=10, t=50, b=10))
    return fig


def short_title(value: str | None, limit: int = 80) -> str:
    if not value:
        return ""
    return value if len(value) <= limit else value[: limit - 1] + "..."


In [7]:

dataset_dir = find_dataset_dir()
train_table = load_arrow_table(dataset_dir / "halohalo-train.arrow")
test_table = load_arrow_table(dataset_dir / "halohalo-test.arrow")

train_rows = train_table.to_pylist()
test_rows = test_table.to_pylist()
all_rows = train_rows + test_rows

print(f"Dataset directory: {dataset_dir}")
print(f"Train rows: {len(train_rows):,}")
print(f"Test rows:  {len(test_rows):,}")
print(f"Total rows: {len(all_rows):,}")
print(f"Columns: {train_table.column_names}")


Dataset directory: /Users/zeraphim/.cache/huggingface/datasets/sapinsapin___halohalo/default/0.0.0/8cf0bd1d6451e552ff95002fdde58eeffe83278f
Train rows: 41,767
Test rows:  3,769
Total rows: 45,536
Columns: ['id', 'text', 'url', 'date', 'dump', 'file_path', 'detected_lang', 'word_count', 'title', 'source', 'language', 'token_count', 'content_hash', 'crawled_at']


In [8]:

patterns = {
    "markdown_heading": re.compile(r"(?m)^#{1,6}\s"),
    "wikipedia": re.compile(r"wikipedia", re.I),
    "researchgate": re.compile(r"researchgate", re.I),
    "slideshare": re.compile(r"slideshare", re.I),
    "boilerplate_published": re.compile(r"This content was originally published by", re.I),
    "pdf_available": re.compile(r"researchpdf available|pdf available", re.I),
}
map_detect = {"tl": "tgl", "hil": "hil", "bik": "bcl"}

train_token_counts = [row["token_count"] for row in train_rows]
test_token_counts = [row["token_count"] for row in test_rows]

language_doc_counts = Counter(row["language"] for row in train_rows)
language_token_counts = Counter()
source_doc_counts = Counter(row["source"] for row in train_rows)
source_token_counts = Counter()
domain_counts_by_source = defaultdict(Counter)
missing_counts = Counter()
pattern_counts = Counter()
pattern_counts_by_source = defaultdict(Counter)
detected_lang_counts = Counter()
detected_lang_mismatches = []

duplicate_hash_counts = Counter(row["content_hash"] for row in train_rows)
train_hashes = set(duplicate_hash_counts)
test_hashes = {row["content_hash"] for row in test_rows}
overlap_hashes = train_hashes & test_hashes

for row in train_rows:
    language_token_counts[row["language"]] += row["token_count"]
    source_token_counts[row["source"]] += row["token_count"]

    url = row.get("url") or ""
    if url:
        domain_counts_by_source[row["source"]][urlparse(url).netloc] += 1

    for field in ["title", "url", "date", "detected_lang", "crawled_at"]:
        value = row.get(field)
        if value is None or value == "":
            missing_counts[field] += 1

    blob = " ".join([
        (row.get("text") or "")[:500],
        row.get("title") or "",
        row.get("url") or "",
    ])
    for name, pattern in patterns.items():
        if pattern.search(blob):
            pattern_counts[name] += 1
            pattern_counts_by_source[row["source"]][name] += 1

    detected = row.get("detected_lang")
    if detected is None:
        detected_lang_counts["none"] += 1
    else:
        mapped = map_detect.get(detected, detected)
        if mapped == row["language"]:
            detected_lang_counts["match"] += 1
        else:
            detected_lang_counts["mismatch"] += 1
            if len(detected_lang_mismatches) < 8:
                detected_lang_mismatches.append({
                    "language": row["language"],
                    "detected_lang": detected,
                    "source": row["source"],
                    "token_count": row["token_count"],
                    "title": short_title(row.get("title")),
                    "url": (row.get("url") or "")[:110],
                })

language_length_stats = {}
for language in sorted(language_doc_counts, key=lambda key: language_doc_counts[key], reverse=True):
    values = [row["token_count"] for row in train_rows if row["language"] == language]
    language_length_stats[language] = summarize_lengths(values)

split_language_counts = {
    "train": Counter(row["language"] for row in train_rows),
    "test": Counter(row["language"] for row in test_rows),
}

train_duplicate_groups = {h: c for h, c in duplicate_hash_counts.items() if c > 1}
duplicate_extra_rows = sum(count - 1 for count in train_duplicate_groups.values())
train_rows_in_duplicate_groups = sum(count for count in train_duplicate_groups.values())

top_duplicate_examples = []
for content_hash, count in duplicate_hash_counts.most_common(8):
    if count <= 1:
        break
    example = next(row for row in train_rows if row["content_hash"] == content_hash)
    top_duplicate_examples.append({
        "count": count,
        "language": example["language"],
        "source": example["source"],
        "token_count": example["token_count"],
        "title": short_title(example.get("title")),
        "url": (example.get("url") or "")[:110],
    })

overlap_examples = []
for content_hash in list(overlap_hashes)[:8]:
    train_example = next(row for row in train_rows if row["content_hash"] == content_hash)
    test_example = next(row for row in test_rows if row["content_hash"] == content_hash)
    overlap_examples.append({
        "language": train_example["language"],
        "source": train_example["source"],
        "token_count": train_example["token_count"],
        "train_title": short_title(train_example.get("title")),
        "test_title": short_title(test_example.get("title")),
        "url": (train_example.get("url") or "")[:110],
    })

suspicious_long_examples = []
for row in sorted(train_rows, key=lambda item: item["token_count"], reverse=True)[:10]:
    suspicious_long_examples.append({
        "language": row["language"],
        "source": row["source"],
        "token_count": row["token_count"],
        "title": short_title(row.get("title"), 90),
        "url": (row.get("url") or "")[:110],
    })

analysis = {
    "train_rows": len(train_rows),
    "test_rows": len(test_rows),
    "total_rows": len(all_rows),
    "train_tokens": int(sum(train_token_counts)),
    "test_tokens": int(sum(test_token_counts)),
    "train_length_summary": summarize_lengths(train_token_counts),
    "test_length_summary": summarize_lengths(test_token_counts),
    "language_doc_counts": language_doc_counts,
    "language_token_counts": language_token_counts,
    "source_doc_counts": source_doc_counts,
    "source_token_counts": source_token_counts,
    "split_language_counts": split_language_counts,
    "language_length_stats": language_length_stats,
    "missing_counts": missing_counts,
    "pattern_counts": pattern_counts,
    "pattern_counts_by_source": pattern_counts_by_source,
    "domain_counts_by_source": domain_counts_by_source,
    "detected_lang_counts": detected_lang_counts,
    "detected_lang_mismatches": detected_lang_mismatches,
    "train_duplicate_group_count": len(train_duplicate_groups),
    "duplicate_extra_rows": duplicate_extra_rows,
    "train_rows_in_duplicate_groups": train_rows_in_duplicate_groups,
    "overlap_hash_count": len(overlap_hashes),
    "train_overlap_rows": sum(1 for row in train_rows if row["content_hash"] in overlap_hashes),
    "test_overlap_rows": sum(1 for row in test_rows if row["content_hash"] in overlap_hashes),
    "top_duplicate_examples": top_duplicate_examples,
    "overlap_examples": overlap_examples,
    "suspicious_long_examples": suspicious_long_examples,
}



## 1. Corpus Overview

The first question is whether this looks like a realistic fine-tuning corpus for Llama 3.1B. That depends less on raw row count and more on token mass, language coverage, and whether the split is representative.


In [9]:

overview_rows = [
    ["train", f"{analysis['train_rows']:,}", f"{analysis['train_tokens']:,}", f"{analysis['train_length_summary']['mean']:.1f}", f"{analysis['train_length_summary']['median']:.1f}", f"{analysis['train_length_summary']['p95']:.1f}", f"{analysis['train_length_summary']['max']:,}"],
    ["test", f"{analysis['test_rows']:,}", f"{analysis['test_tokens']:,}", f"{analysis['test_length_summary']['mean']:.1f}", f"{analysis['test_length_summary']['median']:.1f}", f"{analysis['test_length_summary']['p95']:.1f}", f"{analysis['test_length_summary']['max']:,}"],
]

display(table_figure(
    "Split Summary",
    ["Split", "Rows", "Tokens", "Mean Tokens/Doc", "Median Tokens/Doc", "P95 Tokens/Doc", "Max Tokens/Doc"],
    overview_rows,
    height=240,
))

language_rows = []
for language, doc_count in analysis["language_doc_counts"].most_common():
    token_count = analysis["language_token_counts"][language]
    language_rows.append([
        language,
        f"{doc_count:,}",
        f"{pct(doc_count, analysis['train_rows']):.1f}%",
        f"{token_count:,}",
        f"{pct(token_count, analysis['train_tokens']):.1f}%",
        f"{analysis['language_length_stats'][language]['median']:.0f}",
        f"{analysis['language_length_stats'][language]['p95']:.0f}",
    ])

display(table_figure(
    "Language Mix in Train Split",
    ["Language", "Rows", "Row Share", "Tokens", "Token Share", "Median Tokens/Doc", "P95 Tokens/Doc"],
    language_rows,
    height=350,
))


In [10]:

languages = [lang for lang, _ in analysis["language_doc_counts"].most_common()]
train_lang_counts = [analysis["split_language_counts"]["train"].get(lang, 0) for lang in languages]
test_lang_counts = [analysis["split_language_counts"]["test"].get(lang, 0) for lang in languages]
train_lang_tokens = [analysis["language_token_counts"].get(lang, 0) for lang in languages]

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Train rows by language",
        "Train tokens by language",
        "Train vs test language counts",
        "Top sources by token volume",
    ),
    specs=[[{"type": "bar"}, {"type": "bar"}], [{"type": "bar"}, {"type": "bar"}]],
    vertical_spacing=0.16,
)

fig.add_trace(go.Bar(x=languages, y=train_lang_counts, marker_color="#1f77b4", name="Train rows"), row=1, col=1)
fig.add_trace(go.Bar(x=languages, y=train_lang_tokens, marker_color="#ff7f0e", name="Train tokens"), row=1, col=2)
fig.add_trace(go.Bar(x=languages, y=train_lang_counts, marker_color="#2ca02c", name="Train"), row=2, col=1)
fig.add_trace(go.Bar(x=languages, y=test_lang_counts, marker_color="#d62728", name="Test"), row=2, col=1)

top_sources = analysis["source_token_counts"].most_common(8)
fig.add_trace(
    go.Bar(
        x=[source for source, _ in top_sources],
        y=[tokens for _, tokens in top_sources],
        marker_color="#9467bd",
        name="Source tokens",
    ),
    row=2,
    col=2,
)

fig.update_layout(
    title="Language and Source Distribution",
    height=800,
    width=1100,
    barmode="group",
    showlegend=True,
    margin=dict(l=40, r=20, t=80, b=40),
)
fig.update_xaxes(tickangle=35)
fig



## 2. Sequence Length Pressure

For Llama 3.1B fine-tuning, length distribution matters because it determines packing efficiency, truncation risk, and how aggressively you need to chunk or window documents.


In [11]:

length_thresholds = [128, 256, 512, 1024, 2048, 4096]
threshold_labels = [str(x) for x in length_thresholds]
threshold_counts = [sum(1 for value in train_token_counts if value > threshold) for threshold in length_thresholds]
threshold_share = [pct(count, analysis["train_rows"]) for count in threshold_counts]

box_y = []
box_x = []
for language in languages:
    sample = [row["token_count"] for row in train_rows if row["language"] == language]
    if len(sample) > 2500:
        sample = list(np.random.default_rng(42).choice(sample, size=2500, replace=False))
    box_y.extend(sample)
    box_x.extend([language] * len(sample))

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Token count histogram (linear scale)",
        "Token count histogram (log10 scale)",
        "Rows above context thresholds",
        "Per-language token length spread",
    ),
    specs=[[{"type": "histogram"}, {"type": "histogram"}], [{"type": "bar"}, {"type": "box"}]],
    vertical_spacing=0.16,
)

fig.add_trace(go.Histogram(x=train_token_counts, nbinsx=60, marker_color="#1f77b4", name="token_count"), row=1, col=1)
fig.add_trace(
    go.Histogram(
        x=[math.log10(value) for value in train_token_counts],
        nbinsx=60,
        marker_color="#ff7f0e",
        name="log10(token_count)",
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Bar(
        x=threshold_labels,
        y=threshold_share,
        text=[f"{count:,}" for count in threshold_counts],
        textposition="outside",
        marker_color="#2ca02c",
        name="Rows above threshold (%)",
    ),
    row=2,
    col=1,
)
fig.add_trace(go.Box(x=box_x, y=box_y, marker_color="#9467bd", name="language spread", boxmean=True), row=2, col=2)

fig.update_yaxes(title_text="Rows", row=1, col=1)
fig.update_yaxes(title_text="Rows", row=1, col=2)
fig.update_yaxes(title_text="% of train rows", row=2, col=1)
fig.update_yaxes(title_text="Token count", row=2, col=2)
fig.update_xaxes(title_text="token_count", row=1, col=1)
fig.update_xaxes(title_text="log10(token_count)", row=1, col=2)
fig.update_xaxes(title_text="Threshold", row=2, col=1)
fig.update_layout(title="Document Length Distribution", height=850, width=1100, showlegend=False)
fig



## 3. Quality and Risk Checks

This section targets issues that can quietly hurt training quality or make evaluation misleading: duplicates, train/test leakage, language labeling problems, and low-value scraped pages.


In [12]:

quality_rows = [
    ["Duplicate hash groups in train", f"{analysis['train_duplicate_group_count']:,}", "Exact duplicate content hashes appearing more than once in train."],
    ["Extra duplicate rows in train", f"{analysis['duplicate_extra_rows']:,}", "Rows that could be removed by exact-hash deduplication."],
    ["Train rows inside duplicate groups", f"{analysis['train_rows_in_duplicate_groups']:,}", "Total rows belonging to duplicated documents."],
    ["Overlap hash groups across train/test", f"{analysis['overlap_hash_count']:,}", "Exact content present in both splits."],
    ["Train rows overlapping test hashes", f"{analysis['train_overlap_rows']:,}", "Potential evaluation leakage from train to test."],
    ["Test rows overlapping train hashes", f"{analysis['test_overlap_rows']:,}", "Potential evaluation leakage from test to train."],
    ["Detected language mismatches", f"{analysis['detected_lang_counts']['mismatch']:,}", "Rows where `detected_lang` disagrees with the labeled `language`."],
    ["Missing titles", f"{analysis['missing_counts']['title']:,}", "Metadata completeness signal."],
    ["Missing dates", f"{analysis['missing_counts']['date']:,}", "Metadata completeness signal."],
    ["Missing crawled_at", f"{analysis['missing_counts']['crawled_at']:,}", "Metadata completeness signal."],
]

display(table_figure("Quality Summary", ["Metric", "Value", "Why it matters"], quality_rows, height=520))

noise_rows = []
for name, count in analysis["pattern_counts"].most_common():
    noise_rows.append([name, f"{count:,}", f"{pct(count, analysis['train_rows']):.1f}%"])

display(table_figure("Noise / Boilerplate Pattern Counts", ["Pattern", "Rows", "Share of train"], noise_rows, height=360))


In [13]:

quality_fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Rows in duplicate groups by language",
        "Train/test overlap by split",
        "Noise concentration in top sources",
        "Top web domains in scraped sources",
    ),
    specs=[[{"type": "bar"}, {"type": "bar"}], [{"type": "bar"}, {"type": "bar"}]],
    vertical_spacing=0.16,
)

duplicate_rows_by_language = Counter()
for row in train_rows:
    if duplicate_hash_counts[row["content_hash"]] > 1:
        duplicate_rows_by_language[row["language"]] += 1

quality_fig.add_trace(
    go.Bar(
        x=list(duplicate_rows_by_language.keys()),
        y=list(duplicate_rows_by_language.values()),
        marker_color="#d62728",
        name="Duplicate-group rows",
    ),
    row=1,
    col=1,
)
quality_fig.add_trace(
    go.Bar(
        x=["train rows", "test rows"],
        y=[analysis["train_overlap_rows"], analysis["test_overlap_rows"]],
        marker_color=["#1f77b4", "#ff7f0e"],
        name="Overlap rows",
    ),
    row=1,
    col=2,
)

top_noise_sources = sorted(
    analysis["pattern_counts_by_source"].items(),
    key=lambda item: sum(item[1].values()),
    reverse=True,
)[:6]
quality_fig.add_trace(
    go.Bar(
        x=[source for source, _ in top_noise_sources],
        y=[sum(counter.values()) for _, counter in top_noise_sources],
        marker_color="#2ca02c",
        name="Noise hits",
    ),
    row=2,
    col=1,
)

domain_source = "halo-tgl" if analysis["domain_counts_by_source"].get("halo-tgl") else next(iter(analysis["domain_counts_by_source"]))
top_domains = analysis["domain_counts_by_source"][domain_source].most_common(10)
quality_fig.add_trace(
    go.Bar(
        x=[domain for domain, _ in top_domains],
        y=[count for _, count in top_domains],
        marker_color="#9467bd",
        name=f"Top domains: {domain_source}",
    ),
    row=2,
    col=2,
)

quality_fig.update_layout(title="Quality Risk Signals", height=850, width=1100, showlegend=False)
quality_fig.update_xaxes(tickangle=35)
quality_fig


In [14]:

display(table_figure(
    "Representative Duplicate Examples",
    ["Count", "Language", "Source", "Tokens", "Title", "URL"],
    [[row['count'], row['language'], row['source'], row['token_count'], row['title'], row['url']] for row in analysis['top_duplicate_examples']],
    height=420,
))

display(table_figure(
    "Train/Test Overlap Examples",
    ["Language", "Source", "Tokens", "Train title", "Test title", "URL"],
    [[row['language'], row['source'], row['token_count'], row['train_title'], row['test_title'], row['url']] for row in analysis['overlap_examples']],
    height=420,
))

display(table_figure(
    "Suspiciously Long Documents",
    ["Language", "Source", "Tokens", "Title", "URL"],
    [[row['language'], row['source'], row['token_count'], row['title'], row['url']] for row in analysis['suspicious_long_examples']],
    height=420,
))


In [15]:

train_row_count = analysis["train_rows"]
train_token_count = analysis["train_tokens"]
fil_rows = analysis["language_doc_counts"]["fil"]
hil_tokens = analysis["language_token_counts"]["hil"]
tgl_tokens = analysis["language_token_counts"]["tgl"]
over_2048 = analysis["train_length_summary"]["gt_2048"]
over_4096 = analysis["train_length_summary"]["gt_4096"]

summary_md = f"""
## 4. Key Observations for Llama 3.1B Fine-Tuning

### What this dataset is good for
- This corpus is **better suited for continued pretraining / domain adaptation** than direct instruction tuning.
- Most rows are plain documents, not prompt-response pairs, so using this alone for chat-style SFT would teach the model document continuation more than assistant behavior.

### Scale and composition
- Train split contains **{train_row_count:,} rows** and roughly **{train_token_count:,} tokens**.
- `fil` dominates document count with **{fil_rows:,} rows ({pct(fil_rows, train_row_count):.1f}%)**, but `hil` and `tgl` carry more token mass because their documents are much longer.
- `hil` contributes **{hil_tokens:,} tokens ({pct(hil_tokens, train_token_count):.1f}%)** and `tgl` contributes **{tgl_tokens:,} tokens ({pct(tgl_tokens, train_token_count):.1f}%)**.

### Length profile
- The corpus is highly skewed: median train length is only **{analysis['train_length_summary']['median']:.0f} tokens**, while the mean is **{analysis['train_length_summary']['mean']:.0f}**.
- **{over_2048:,} train rows ({pct(over_2048, train_row_count):.1f}%)** exceed 2,048 tokens.
- **{over_4096:,} train rows ({pct(over_4096, train_row_count):.1f}%)** exceed 4,096 tokens.
- You should plan to **chunk long documents** before training rather than feed many full pages directly.

### Quality risks
- There are **{analysis['train_duplicate_group_count']:,} duplicate hash groups** in train, creating **{analysis['duplicate_extra_rows']:,} removable extra rows**.
- There are **{analysis['overlap_hash_count']:,} exact content hashes shared across train and test**, affecting **{analysis['test_overlap_rows']:,} test rows**. This makes the current test split unreliable for clean evaluation.
- The test split is also not representative: it is heavily `fil`/`hil` and appears to omit `tgl` and `bcl` entirely.
- `detected_lang` disagrees with the labeled language on **{analysis['detected_lang_counts']['mismatch']:,} rows**, which suggests either label noise or domain drift in scraped pages.

### Scraping noise to clean up first
- The corpus contains many obvious low-value scraper artifacts, including **{analysis['pattern_counts']['wikipedia']:,} Wikipedia-like rows**, **{analysis['pattern_counts']['researchgate']:,} ResearchGate-like rows**, and **{analysis['pattern_counts']['boilerplate_published']:,} publisher boilerplate rows**.
- Several longest or duplicated samples are clearly noisy, such as login pages, feed pages, slideshow fragments, or cross-language Wikipedia content.

### Practical recommendation
1. Deduplicate by `content_hash` before any training run.
2. Rebuild the eval split after deduplication and make sure every target language is represented.
3. Filter obvious boilerplate domains and page types such as login, edit, feed, slideshow, and other navigational pages.
4. Chunk long documents to your target context window, for example 1k to 2k tokens with overlap if you want continued pretraining.
5. Consider language-aware sampling because row count and token count tell different stories here.
6. For the actual project plan, use this corpus for **continued pretraining first**, then add a separate instruction dataset for chat-style behavior.
"""

display(Markdown(summary_md))



## 4. Key Observations for Llama 3.1B Fine-Tuning

### What this dataset is good for
- This corpus is **better suited for continued pretraining / domain adaptation** than direct instruction tuning.
- Most rows are plain documents, not prompt-response pairs, so using this alone for chat-style SFT would teach the model document continuation more than assistant behavior.

### Scale and composition
- Train split contains **41,767 rows** and roughly **24,666,596 tokens**.
- `fil` dominates document count with **24,992 rows (59.8%)**, but `hil` and `tgl` carry more token mass because their documents are much longer.
- `hil` contributes **9,332,784 tokens (37.8%)** and `tgl` contributes **8,208,749 tokens (33.3%)**.

### Length profile
- The corpus is highly skewed: median train length is only **218 tokens**, while the mean is **591**.
- **2,097 train rows (5.0%)** exceed 2,048 tokens.
- **1,198 train rows (2.9%)** exceed 4,096 tokens.
- You should plan to **chunk long documents** before training rather than feed many full pages directly.

### Quality risks
- There are **2,563 duplicate hash groups** in train, creating **9,797 removable extra rows**.
- There are **521 exact content hashes shared across train and test**, affecting **704 test rows**. This makes the current test split unreliable for clean evaluation.
- The test split is also not representative: it is heavily `fil`/`hil` and appears to omit `tgl` and `bcl` entirely.
- `detected_lang` disagrees with the labeled language on **7,394 rows**, which suggests either label noise or domain drift in scraped pages.

### Scraping noise to clean up first
- The corpus contains many obvious low-value scraper artifacts, including **5,143 Wikipedia-like rows**, **1,280 ResearchGate-like rows**, and **4,780 publisher boilerplate rows**.
- Several longest or duplicated samples are clearly noisy, such as login pages, feed pages, slideshow fragments, or cross-language Wikipedia content.

### Practical recommendation
1. Deduplicate by `content_hash` before any training run.
2. Rebuild the eval split after deduplication and make sure every target language is represented.
3. Filter obvious boilerplate domains and page types such as login, edit, feed, slideshow, and other navigational pages.
4. Chunk long documents to your target context window, for example 1k to 2k tokens with overlap if you want continued pretraining.
5. Consider language-aware sampling because row count and token count tell different stories here.
6. For the actual project plan, use this corpus for **continued pretraining first**, then add a separate instruction dataset for chat-style behavior.
